# GuardSLM: 100-Pair (200 Cases) LMS Action Authorization Benchmark
### Context-Aware Permission Gatekeeping for Browser-Based AI Agents

This notebook provides the complete, reproducible evaluation pipeline for the canonical **GuardSLM 100-Pair (200 Cases) Benchmark** (`GuardSLM_100_Pairs_v2.json`).

### Strategy & Model Modality Support Matrix

| Model Name | Model Type | Parameters | Strategy A (Text-Only) | Strategy B (Text + Screenshot) | Notes |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **PolicyGuard-4B** | Trajectory Safety Guard | 4.0B | ✅ **Supported** | ❌ *Unsupported* | Native web agent trajectory format (`Step 1: Request, Step 2: Role, Step 3: State, Step 4: Action` → `violation` / `no_violation`). Evaluated on 24-case subset (12 matched pairs). |
| **DynaGuard-1.7B** | Policy Compliance Gatekeeper | 1.7B | ✅ **Supported** | ❌ *Unsupported* | Native rule compliance format (`USER ROLE, REQUEST, CONTEXT, POLICY, PROPOSED ACTION` → `ALLOW` / `BLOCK` / `ASK`). Evaluated on 24-case subset (12 matched pairs). |
| **WebGuard-3B** | Action Risk Guard | 3.0B | ✅ **Supported** | ❌ *Unsupported* | Native accessibility-tree consequence format (`{"risky_level": "SAFE" | "LOW" | "HIGH"}`). Evaluated on 24-case subset (12 matched pairs). |
| **Qwen3-VL 2B Instruct** | Multimodal VLM | 2.0B | ✅ **Supported** | ✅ **Supported** | **Multimodal Vision-Language Model.** Evaluates both Strategy A and B to measure visual grounding delta. |
| **Mock Benchmark Guard** | Pipeline Simulation | 0 | ✅ **Supported** | ✅ **Supported** | Instant CPU test harness for verification. |

---


### Step 1 — Environment Setup & Path Configuration
Clone the repository (if on Google Colab) or configure workspace working directory.

In [ ]:
import os, sys

# Check if running inside Google Colab
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo_dir = "/content/GuardSLM"
    if not os.path.exists(repo_dir):
        !git clone https://github.com/AkarshiAaryan/GuardSLM.git {repo_dir}
    os.chdir(repo_dir)
    print("Running inside Google Colab! Changed working directory to:", os.getcwd())
else:
    print("Running locally in working directory:", os.getcwd())

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())


### Step 2 — Install Dependencies & Poppler Utilities
Install PyTorch, Hugging Face Transformers, BitsAndBytes, and poppler utilities for lossless PDF screenshot rendering.

In [ ]:
!apt-get -qq update && apt-get -qq install -y poppler-utils
!pip install -q -U git+https://github.com/huggingface/transformers.git accelerate torchvision qwen-vl-utils
!pip install -q -r requirements.txt
print("Dependencies successfully installed!")


### Step 3 — Hardware & Accelerator Check
Verify CUDA GPU availability, device name, and total VRAM.

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"CUDA Accelerator Available: {device_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("WARNING: No CUDA GPU detected. Models will run on CPU or mock mode.")


### Step 4 — Structural Validation of Canonical Dataset
Run the programmatic validator on `data/GuardSLM_100_Pairs_v2.json` to verify the 13 structural invariants (36 policies, 200 cases, 100 pairs, 80 flips, 20 invariants).

In [ ]:
from src.benchmark.data.validator import validate_benchmark_dataset, generate_markdown_validation_report
from IPython.display import display, Markdown

val_res = validate_benchmark_dataset("data/GuardSLM_100_Pairs_v2.json")
report_text = generate_markdown_validation_report(val_res, "reports/guardslm_100_pairs_validation_report.md")
display(Markdown(report_text))
assert val_res.is_valid, "Dataset failed structural validation!"


### Step 5 — Screenshot Resolution & Extraction for Strategy B
Extract all unique Moodle screenshot pages referenced across the benchmark losslessly into `data/extracted_screenshots/`.

In [ ]:
from src.benchmark.data.loader import BenchmarkLoader
from src.benchmark.vision.screenshot_resolver import ScreenshotResolver

loader = BenchmarkLoader()
cases = loader.get_model_inputs()
resolver = ScreenshotResolver()

resolved_count = 0
for case in cases:
    path, meta = resolver.resolve_case_screenshot(case)
    if meta.get("status") == "resolved":
        resolved_count += 1

print(f"Successfully resolved original Moodle screenshots for {resolved_count}/{len(cases)} cases!")


### Step 6 — Fast Pipeline Smoke Test (Mock Guard on Both Strategies)
Execute the complete pipeline on a small 4-case subset to verify that prompt rendering, strict parsing, resumption, and evaluation reporting execute cleanly.

In [ ]:
from src.benchmark.runner.experiment_runner import run_benchmark_experiment

test_results = run_benchmark_experiment(
    dataset_path="data/GuardSLM_100_Pairs_v2.json",
    model_name="mock_guard",
    model_config={"mode": "perfect"},
    strategy="both",
    limit=4,
    output_dir="data/benchmark_results/smoke_test"
)
print("Smoke test completed successfully!")
print("Strategy A Accuracy:", test_results["text_only"]["decision_performance"]["accuracy"])
print("Strategy B Accuracy:", test_results["text_plus_screenshot"]["decision_performance"]["accuracy"])


### Step 7A — Specialized Guard 1: PolicyGuard-4B (Trajectory Safety Guard)
Evaluate **PolicyGuard-4B** (`Rakancorle1/PolicyGuard-4B`) using its native web agent trajectory format (`Step 1: Request, Step 2: Role, Step 3: State, Step 4: Action` → `violation` / `no_violation`).
Evaluated on a representative 24-case subset (12 complete matched pairs: `limit=24`).

In [ ]:
policyguard_results = run_benchmark_experiment(
    dataset_path="data/GuardSLM_100_Pairs_v2.json",
    model_name="policyguard",
    model_config={"checkpoint": "Rakancorle1/PolicyGuard-4B", "load_in_4bit": False},
    strategy="text_only",
    limit=24,
    resume=True
)
print("PolicyGuard Decision Accuracy:", policyguard_results["text_only"]["decision_performance"]["accuracy"])
print("PolicyGuard Pair Sensitivity:", policyguard_results["text_only"]["matched_pair_performance"]["authorization_flip_accuracy"])


### Step 7B — Specialized Guard 2: DynaGuard-1.7B (Policy Compliance Gatekeeper)
Evaluate **DynaGuard-1.7B** (`tomg-group-umd/DynaGuard-1.7B`) using its native rule-following gatekeeper format (`USER ROLE, USER REQUEST, CURRENT CONTEXT, RELEVANT POLICY, PROPOSED ACTION` → `ALLOW` / `BLOCK` / `ASK`).
Evaluated on a representative 24-case subset (12 complete matched pairs: `limit=24`).

In [ ]:
dynaguard_results = run_benchmark_experiment(
    dataset_path="data/GuardSLM_100_Pairs_v2.json",
    model_name="dynaguard",
    model_config={"checkpoint": "tomg-group-umd/DynaGuard-1.7B", "load_in_4bit": False},
    strategy="text_only",
    limit=24,
    resume=True
)
print("DynaGuard Decision Accuracy:", dynaguard_results["text_only"]["decision_performance"]["accuracy"])
print("DynaGuard Pair Sensitivity:", dynaguard_results["text_only"]["matched_pair_performance"]["authorization_flip_accuracy"])


### Step 7C — Specialized Guard 3: WebGuard-3B (Accessibility Tree Action Guard)
Evaluate **WebGuard-3B** (`Qwen/Qwen2.5-3B-Instruct` base or with LoRA adapter) using its native accessibility-tree consequence format (`{"risky_level": "SAFE" | "LOW" | "HIGH"}`).
Evaluated on a representative 24-case subset (12 complete matched pairs: `limit=24`).

In [ ]:
webguard_results = run_benchmark_experiment(
    dataset_path="data/GuardSLM_100_Pairs_v2.json",
    model_name="webguard",
    model_config={"checkpoint": "Qwen/Qwen2.5-3B-Instruct", "adapter_path": "models/webguard_qwen_lora", "augmented": True, "load_in_4bit": False},
    strategy="text_only",
    limit=24,
    resume=True
)
print("WebGuard Decision Accuracy:", webguard_results["text_only"]["decision_performance"]["accuracy"])
print("WebGuard Pair Sensitivity:", webguard_results["text_only"]["matched_pair_performance"]["authorization_flip_accuracy"])


### Step 8A — Multimodal Model: Qwen3-VL 2B (Strategy A: Text-Only Baseline)
Evaluate the latest Qwen3-VL 2B Multimodal model in **Strategy A (Text-only)** without screenshots to establish its text baseline.

In [ ]:
qwen3_vl_strat_a = run_benchmark_experiment(
    dataset_path="data/GuardSLM_100_Pairs_v2.json",
    model_name="qwen3_vl_2b",
    model_config={"checkpoint": "Qwen/Qwen3-VL-2B-Instruct", "is_vlm": True, "load_in_4bit": False},
    strategy="text_only",
    resume=True
)


### Step 8B — Multimodal Model: Qwen3-VL 2B (Strategy B: Text + Original Screenshot)
Evaluate **Qwen3-VL 2B** under **Strategy B (Text + Verified Moodle Screenshot)**.
Comparing Step 8B against Step 8A directly measures the exact impact of visual UI grounding vs text-only grounding on the identical model.

In [ ]:
qwen3_vl_strat_b = run_benchmark_experiment(
    dataset_path="data/GuardSLM_100_Pairs_v2.json",
    model_name="qwen3_vl_2b",
    model_config={"checkpoint": "Qwen/Qwen3-VL-2B-Instruct", "is_vlm": True, "load_in_4bit": False},
    strategy="text_plus_screenshot",
    resume=True
)


### Step 9 — Consolidated Benchmark Report & Decision Gate Verdict
Synthesize side-by-side comparative performance tables across all models and strategies.

In [ ]:
import glob
from IPython.display import display, Markdown

report_files = sorted(glob.glob("data/benchmark_results/*/report_*.md"))
print(f"Discovered {len(report_files)} benchmark reports:")
for r in report_files:
    print(" - ", r)

if report_files:
    with open(report_files[-1], "r", encoding="utf-8") as f:
        display(Markdown(f.read()))
